In [9]:
# STEP 1 — Imports, Config (YAML), and Helpers
import os, re
from pathlib import Path
from datetime import datetime

import yaml
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
import pypandoc  # Markdown → DOCX

# --- LangChain Core ---
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import Ollama

load_dotenv()

# ---------- Resolve project root (works in notebook or script) ----------
try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    PROJECT_ROOT = Path.cwd().parent

# ---------- Notebook-only output root ----------
NOTEBOOK_DIR = Path.cwd()
NB_OUT_ROOT = NOTEBOOK_DIR / "3-Output_experiemnt_RAG_v3_NIH_all"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = NB_OUT_ROOT / f"run_{RUN_ID}"

NB_MD_DIR   = RUN_DIR / "md"
NB_DOCX_DIR = RUN_DIR / "docx"

for p in [NB_OUT_ROOT, RUN_DIR, NB_MD_DIR, NB_DOCX_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ---------- YAML loader + path resolver ----------
def load_yaml_config(cfg_path: Path) -> dict:
    cfg_path = Path(cfg_path)
    if not cfg_path.exists():
        raise FileNotFoundError(f"Config YAML not found: {cfg_path}")
    with cfg_path.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise ValueError(f"Config YAML must parse to a dict. Got: {type(cfg)}")
    return cfg

def resolve_from_root(project_root: Path, root_dir_value: str | Path) -> Path:
    p = Path(root_dir_value).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (project_root / p).resolve()

def resolve_path(base: Path, rel_or_abs: str | Path | None) -> Path | None:
    if rel_or_abs is None:
        return None
    p = Path(rel_or_abs).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (base / p).resolve()

def _ensure_bool(name: str, v):
    if not isinstance(v, bool):
        raise TypeError(
            f"{name} must be a YAML boolean true/false (no quotes). Got {v!r} ({type(v)})"
        )
    return v

# ---------- Choose your YAML file here ----------
CONFIG_YAML = PROJECT_ROOT / "config" / "config.yaml"
cfg = load_yaml_config(CONFIG_YAML)

# ---------- Root dir from YAML ----------
ROOT_DIR = resolve_from_root(PROJECT_ROOT, cfg["root_dir"])

# ---------- Paths from YAML (READ-ONLY / pipeline assets) ----------
DATA_PDFS  = resolve_path(ROOT_DIR, cfg["paths"]["data_pdfs"])
INDEX_DIR  = resolve_path(ROOT_DIR, cfg["paths"]["index_dir"])
EXCEL_PATH = resolve_path(ROOT_DIR, cfg["paths"]["excel_path"])

TEMPLATE_MD = resolve_path(
    ROOT_DIR,
    cfg["paths"].get("template_md", "data/inputs/dmp-template.md")
)

# ---------- RAG params ----------
TOP_K = int(cfg["rag"]["retriever_top_k"])

# ---------- Models ----------
EMBED_MODEL = cfg["models"]["embedding_model"]
LLM_MODEL   = cfg["models"]["llm_name"]

EMBED_DEVICE     = cfg["models"]["embedding_device"]
EMBED_BATCH_SIZE = int(cfg["models"]["embedding_batch_size"])

NORMALIZE_EMBEDS = _ensure_bool("models.normalize_embeddings", cfg["models"]["normalize_embeddings"])
LOCAL_FILES_ONLY = _ensure_bool("models.local_files_only", cfg["models"]["local_files_only"])
ALLOW_DL_IF_MISS = _ensure_bool("models.allow_download_if_missing", cfg["models"]["allow_download_if_missing"])

HF_CACHE_DIR = resolve_path(ROOT_DIR, cfg["models"]["hf_cache_dir"])

# ---------- Notebook-only outputs ----------
OUTPUT_MD   = NB_MD_DIR / "generated_dmp.md"
OUTPUT_DOCX = NB_DOCX_DIR / "generated_dmp.docx"

# ---------- Helper functions ----------
def create_folder(folderpath: Path | str) -> None:
    Path(folderpath).mkdir(parents=True, exist_ok=True)

def sanitize_filename(name: str, max_len: int = 140) -> str:
    s = re.sub(r'[\\/*?:"<>|]', "_", str(name).strip())
    s = re.sub(r"\s+", " ", s).strip()
    if len(s) > max_len:
        s = s[:max_len].rstrip()
    return s or "untitled"

def save_md(folderpath: Path | str, filename: str, text: str) -> Path:
    create_folder(folderpath)
    out_path = Path(folderpath) / filename
    out_path.write_text(text, encoding="utf-8")
    return out_path

def md_to_docx(md_filepath: Path | str, docx_folderpath: Path | str, docx_filename: str) -> Path:
    create_folder(docx_folderpath)
    out_path = Path(docx_folderpath) / docx_filename
    try:
        pypandoc.get_pandoc_version()
    except OSError as e:
        raise RuntimeError(
            "Pandoc is not available. Install it, or run:\n"
            "  pypandoc.download_pandoc()\n"
            f"Original error: {e}"
        )
    pypandoc.convert_file(str(md_filepath), "docx", outputfile=str(out_path))
    return out_path

# ---------- Existence checks (fail early) ----------
for label, p in {
    "CONFIG_YAML": CONFIG_YAML,
    "INDEX_DIR": INDEX_DIR,
    "EXCEL_PATH": EXCEL_PATH,
    "TEMPLATE_MD": TEMPLATE_MD,
}.items():
    if p is None or not Path(p).exists():
        raise FileNotFoundError(f"{label} does not exist: {p}")

print("STEP 1 ready")
print("CONFIG_YAML :", CONFIG_YAML)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ROOT_DIR    :", ROOT_DIR)
print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("RUN_DIR     :", RUN_DIR)
print("INDEX_DIR   :", INDEX_DIR)
print("DATA_PDFS   :", DATA_PDFS)
print("EXCEL_PATH  :", EXCEL_PATH)
print("TEMPLATE_MD :", TEMPLATE_MD)
print("LLM_MODEL   :", LLM_MODEL)
print("TOP_K       :", TOP_K)

STEP 1 ready
CONFIG_YAML : c:\Users\Nahid\dmpchef\config\config.yaml
PROJECT_ROOT: c:\Users\Nahid\dmpchef
ROOT_DIR    : C:\Users\Nahid\dmpchef
NOTEBOOK_DIR: c:\Users\Nahid\dmpchef\notebook_DMP_RAG
RUN_DIR     : c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748
INDEX_DIR   : C:\Users\Nahid\dmpchef\data\vector_db\DMPtools_db
DATA_PDFS   : C:\Users\Nahid\dmpchef\data\data_ingestion\NIH_DMPtools
EXCEL_PATH  : C:\Users\Nahid\dmpchef\data\inputs\inputs.xlsx
TEMPLATE_MD : C:\Users\Nahid\dmpchef\data\inputs\dmp-template.md
LLM_MODEL   : llama3.3:70b
TOP_K       : 6


In [2]:
# STEP 2 — Load pipeline FAISS index + retriever
from pathlib import Path
import os
import torch

# Prefer new HuggingFace integration if installed; fallback to langchain_community
try:
    from langchain_huggingface import HuggingFaceEmbeddings  # type: ignore
    _EMB_BACKEND = "langchain_huggingface"
except Exception:
    from langchain_community.embeddings import HuggingFaceEmbeddings  # type: ignore
    _EMB_BACKEND = "langchain_community"

# Ensure HF uses same cache directory
if HF_CACHE_DIR is not None:
    os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))

def _pick_device(requested: str) -> str:
    req = (requested or "auto").lower().strip()
    if req in ("auto", "cuda"):
        return "cuda" if torch.cuda.is_available() else "cpu"
    return "cpu"

device = _pick_device(EMBED_DEVICE)

def _make_embeddings(local_only: bool):
    return HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        cache_folder=str(HF_CACHE_DIR) if HF_CACHE_DIR is not None else None,
        model_kwargs={"device": device, "local_files_only": bool(local_only)},
        encode_kwargs={"batch_size": int(EMBED_BATCH_SIZE), "normalize_embeddings": bool(NORMALIZE_EMBEDS)},
    )

# Try local-only first, then fallback if allowed
try:
    embeddings = _make_embeddings(local_only=LOCAL_FILES_ONLY)
except Exception as e1:
    if LOCAL_FILES_ONLY and ALLOW_DL_IF_MISS:
        print("Embeddings not found in cache; retrying with download enabled...")
        embeddings = _make_embeddings(local_only=False)
    else:
        raise

index_dir = Path(INDEX_DIR)
faiss_path = index_dir / "index.faiss"
pkl_path   = index_dir / "index.pkl"

if not (faiss_path.exists() and pkl_path.exists()):
    raise FileNotFoundError(
        "FAISS index files not found.\n"
        f"Expected:\n- {faiss_path}\n- {pkl_path}\n"
    )

print("Loading FAISS index from:", index_dir)
vectorstore = FAISS.load_local(
    str(index_dir),
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": int(TOP_K)})

print(
    "Retriever ready",
    f"top_k={TOP_K}",
    f"embed_model={EMBED_MODEL}",
    f"device={device}",
    f"backend={_EMB_BACKEND}",
    sep=" | "
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1537.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FAISS index from: C:\Users\Nahid\dmpchef\data\vector_db\NIH_all_db
Retriever ready | top_k=6 | embed_model=sentence-transformers/all-MiniLM-L6-v2 | device=cuda | backend=langchain_huggingface


In [3]:
# STEP 3 — Load Excel + Template + Build RAG Chain
# --- Guards ---
if "retriever" not in globals():
    raise RuntimeError("`retriever` is not defined. Run STEP 2 first.")

# --- Load Excel ---
df = pd.read_excel(EXCEL_PATH)
df.columns = df.columns.str.strip().str.lower()
df = df.fillna("")
print(f"Excel loaded: {len(df)} rows")

# --- Load template ---
dmp_template_text = Path(TEMPLATE_MD).read_text(encoding="utf-8")
print("Template loaded.")

def format_docs(docs):
    if not docs:
        return ""
    formatted = []
    for d in docs:
        page = d.metadata.get("page", d.metadata.get("page_number", ""))
        source = d.metadata.get("source", d.metadata.get("file_path", ""))
        page_disp = (page + 1) if isinstance(page, int) else page
        formatted.append(f"[Page {page_disp}] {source}\n{(d.page_content or '').strip()}")
    return "\n\n".join(formatted)

PROMPT_TMPL = """You are an expert biomedical grant writer and data steward.

Create a Data Management and Sharing Plan (DMSP) for an NIH grant.

---- Retrieved NIH Policy Context ----
{context}

Specifically targeting the {institute}.

Here are the details about the data to be collected:
{element_1a}

{consent_block}

Provide the result using exactly this markdown format template of the DMSP provided by the NIH without changing it:

{template_md}

"""

prompt = PromptTemplate(
    template=PROMPT_TMPL,
    input_variables=["context", "institute", "element_1a", "consent_block", "template_md"]
)

llm = Ollama(model=LLM_MODEL)
parser = StrOutputParser()

# IMPORTANT: chain input is a dict, and retrieval uses query_for_retrieval from that dict
rag_chain = (
    {
        "context": (lambda x: x["query_for_retrieval"]) | retriever | format_docs,
        "institute": lambda x: x["institute"],
        "element_1a": lambda x: x["element_1a"],
        "consent_block": lambda x: x["consent_block"],
        "template_md": lambda x: x["template_md"],
    }
    | prompt
    | llm
    | parser
)

print("RAG chain ready.")

Excel loaded: 26 rows
Template loaded.
RAG chain ready.


C:\Users\Nahid\AppData\Local\Temp\ipykernel_46052\3316586724.py:52: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model=LLM_MODEL)


In [4]:
# STEP 3.1 — Build chain input from one row
def build_chain_input_from_row(row: pd.Series) -> dict:
    institute = str(row.get("institute", "")).strip()
    element_1a = str(row.get("element_1a", "")).strip()

    is_human = str(row.get("ishumanstudy", "")).strip().lower()
    consent_desc = str(row.get("consentdescription", "")).strip()

    if "yes" in is_human:
        consent_block = (
            "This proposal includes a study that will involve human participants. "
            + consent_desc
        ).strip()
    else:
        consent_block = ""

    query_for_retrieval = "\n".join([institute, element_1a, consent_block]).strip()

    return {
        "query_for_retrieval": query_for_retrieval,
        "institute": institute,
        "element_1a": element_1a,
        "consent_block": consent_block,
        "template_md": dmp_template_text,
    }

In [5]:
# STEP 4 — Generate DMPs for all rows (RAG)

# --- Guards ---
if "rag_chain" not in globals():
    raise RuntimeError("rag_chain not found. Run STEP 3 first.")
if "df" not in globals():
    raise RuntimeError("df not found. Run STEP 3 first.")

OUTPUT_LOG = RUN_DIR / "rag_generated_dmp_log.csv"

run_id = RUN_DIR.name
started_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

records = []

# NEW: control how many rows to generate
df_to_generate = df

for idx, row in tqdm(df_to_generate.iterrows(), total=len(df_to_generate), desc="Generating NIH DMPs (RAG)"):
    # Prefer exact Excel title (so filenames align with inputs.xlsx)
    title = str(row.get("title", "")).strip()

    # Fallbacks only if title missing/empty
    if not title:
        title = str(row.get("designation", "")).strip()
    if not title:
        title = str(row.get("institute", "")).strip()
    if not title:
        title = f"Project_{idx+1:03d}"

    safe_title = sanitize_filename(title)

    chain_input = build_chain_input_from_row(row)

    try:
        response = rag_chain.invoke(chain_input)  # dict input

        md_filename = f"{safe_title}.md"
        docx_filename = f"{safe_title}.docx"

        md_path = save_md(NB_MD_DIR, md_filename, response)
        docx_path = md_to_docx(md_path, NB_DOCX_DIR, docx_filename)

        records.append({
            "run_id": run_id,
            "started_at": started_at,
            "row_index": idx,
            "title": title,
            "md_filename": md_filename,
            "docx_filename": docx_filename,
            "md_path": str(md_path),
            "docx_path": str(docx_path),
            "query_for_retrieval_preview": chain_input.get("query_for_retrieval", "")[:800],
            "generated_preview": str(response)[:800],
            "error": "",
        })

    except Exception as e:
        records.append({
            "run_id": run_id,
            "started_at": started_at,
            "row_index": idx,
            "title": title,
            "md_filename": "",
            "docx_filename": "",
            "md_path": "",
            "docx_path": "",
            "query_for_retrieval_preview": chain_input.get("query_for_retrieval", "")[:800],
            "generated_preview": "",
            "error": repr(e),
        })

pd.DataFrame(records).to_csv(OUTPUT_LOG, index=False, encoding="utf-8")
print("Done. Log saved to:", OUTPUT_LOG)
print("MD outputs :", NB_MD_DIR)
print("DOCX outputs:", NB_DOCX_DIR)

Generating NIH DMPs (RAG): 100%|██████████| 26/26 [4:24:09<00:00, 609.60s/it]  

Done. Log saved to: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_061954\rag_generated_dmp_log.csv
MD outputs : c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_061954\md
DOCX outputs: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_061954\docx


In [17]:
# ==========================================================
# STEP 5–8 — Evaluation 
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer


# Generated markdowns (from the same run you just created in Step 1–4)
MODEL_DIR = Path(NB_MD_DIR)

# Evaluation outputs inside the same run folder
EVAL_DIR = Path(RUN_DIR) / "evaluation_results"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_DIR (md):", MODEL_DIR, "| exists=", MODEL_DIR.exists())
print("EVAL_DIR:", EVAL_DIR)

MODEL_DIR (md): c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\md | exists= True
EVAL_DIR: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results


In [18]:
HUMAN_XLSX = Path(EXCEL_PATH)     # gold reference is your inputs.xlsx
MODEL_DIR  = Path(NB_MD_DIR)      

# Step 5 outputs
RAG_OUT_CSV = EVAL_DIR / "filtered_RAG.csv"
# Step 6 outputs
RAG_MERGED  = EVAL_DIR / "merged_output_RAG.csv"
# Step 7 outputs
RAG_CLEANED = EVAL_DIR / "merged_output_RAG_cleaned.csv"
# Step 8 outputs
OUT_FOLDER = EVAL_DIR / "dmp_similarity_summary_RAG.csv"
OUT_RAW = EVAL_DIR / "element_similarity_raw_RAG.csv"
OUT_ELEMENT = EVAL_DIR / "element_similarity_summary_RAG.csv"
OUT_SUB_ELEMENT = EVAL_DIR / "sub_element_similarity_summary_RAG.csv"

print("Gold Excel:", HUMAN_XLSX, "| exists=", HUMAN_XLSX.exists())
print("Generated MD dir:", MODEL_DIR, "| exists=", MODEL_DIR.exists())
print("Eval output dir:", EVAL_DIR)

if not HUMAN_XLSX.exists():
    raise FileNotFoundError(f"Gold Excel not found: {HUMAN_XLSX}")
if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Generated Markdown folder not found: {MODEL_DIR}")

md_files = list(MODEL_DIR.glob("*.md"))
print("Found generated .md:", len(md_files))
if not md_files:
    raise FileNotFoundError(f"No generated .md files found in: {MODEL_DIR}")


# -----------------------------
# STEP 2 — Shared Utilities (same as your version)
# -----------------------------
def safe_text(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def norm_text(x) -> str:
    s = safe_text(x).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def choose_title_column(df: pd.DataFrame) -> str:
    candidates = ["title", "Title", "dmp_title", "DMP Title", "DMP_title"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find title column. Columns: {list(df.columns)}")

def sort_element_key(x: str):
    s = str(x).strip().lower()
    m = re.match(r"^element_(\d+)([a-z]?)$", s)
    if not m:
        return (999, "z", s)
    return (int(m.group(1)), m.group(2) or "", s)

def rougeL_recall(ref: str, gen: str, scorer: rouge_scorer.RougeScorer) -> float:
    ref = ref or ""
    gen = gen or ""
    if not ref.strip() and not gen.strip():
        return 1.0
    if not ref.strip() or not gen.strip():
        return 0.0
    return float(scorer.score(ref, gen)["rougeL"].recall)


# -----------------------------
# STEP 3 — Markdown Parsing Helpers (same as your version)
# -----------------------------
def clean_title_for_match(name: str) -> str:
    s = str(name).strip()
    s = re.sub(r'[\\/*?:"<>|]', "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def normalize_md_stem(stem: str) -> str:
    s = (stem or "").strip().lower()
    s = re.sub(r"[-_\s]?gpt[-_\s]?[\d\.]+$", "", s)
    s = re.sub(r"[-_\s]?llama[-_\s]?[\d\.]+$", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _canon_keep_nia(s: str) -> str:
    """
    Canonical form:
    - normalize separators (_ and - -> space)
    - DO NOT remove 'NIA'
    - remove trailing model suffix only (gpt / gpt4.1 / llama / llama3.3)
    """
    s = safe_text(s).strip().lower()
    s = s.replace("_", " ").replace("-", " ")
    s = re.sub(r"\s+", " ", s).strip()

    # remove ONLY model suffix at the end
    s = re.sub(r"\b(gpt|llama)\s*[\d\.]*\s*$", "", s, flags=re.I).strip()

    # keep NIA
    s = re.sub(r"\s+", " ", s).strip()
    return s

def find_md_by_excel_title_v3(search_dir: Path, excel_title: str) -> Path | None:
    excel_norm = _canon_keep_nia(clean_title_for_match(excel_title))
    md_files = list(search_dir.rglob("*.md"))
    if not md_files:
        return None

    for p in md_files:
        stem_norm = _canon_keep_nia(p.stem)
        if stem_norm == excel_norm:
            return p

    for p in md_files:
        stem_norm = _canon_keep_nia(p.stem)
        if stem_norm == f"{excel_norm} nia":
            return p

    candidates = []
    for p in md_files:
        stem_norm = _canon_keep_nia(p.stem)

        if excel_norm and (excel_norm in stem_norm or stem_norm in excel_norm):
            candidates.append((abs(len(stem_norm) - len(excel_norm)), len(p.stem), str(p), p))

        if excel_norm and stem_norm.endswith(" nia") and excel_norm in stem_norm[:-4]:
            candidates.append((abs(len(stem_norm) - len(excel_norm)), len(p.stem), str(p), p))

    if candidates:
        candidates.sort(key=lambda x: (x[0], x[1], x[2]))
        return candidates[0][3]

    return None

def is_title(line: str) -> bool:
    s = (line or "").strip()
    if not s:
        return False
    if s.startswith("#"):
        return True
    if re.match(r"^\s*\d+[\.\)]?\s*\*\*.+\*\*\s*:?\s*$", s):
        return True
    if re.match(r"^\s*\*\*\s*element\s*\d+\s*:\s*.+\*\*\s*:?\s*$", s, flags=re.I):
        return True
    return False

def extract_titles_and_text(md_text: str, content_col: str) -> pd.DataFrame:
    cleaned = re.sub(r"<think>.*?</think>", "", md_text, flags=re.DOTALL | re.IGNORECASE)
    lines = cleaned.splitlines()

    rows = []
    current_title = None
    buf = []

    def flush():
        nonlocal current_title, buf, rows
        if current_title is None:
            return
        text = "\n".join(buf).strip()
        if text:
            rows.append({"Element title": current_title.strip(), content_col: text})

    for line in lines:
        if is_title(line):
            flush()
            current_title = line
            buf = []
        else:
            buf.append(line)

    flush()
    return pd.DataFrame(rows)

Gold Excel: C:\Users\Nahid\dmpchef\data\inputs\inputs.xlsx | exists= True
Generated MD dir: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\md | exists= True
Eval output dir: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results
Found generated .md: 26


In [19]:
# STEP 4 — Section Title → Element Mapping (same structure)
def map_section_title_to_element(section_title) -> str | None:
    if section_title is None:
        return None
    try:
        if pd.isna(section_title):
            return None
    except Exception:
        pass

    t = str(section_title).strip()
    t_plain = re.sub(r"^#+\s*", "", t).strip()
    t_low = t_plain.lower()

    if re.search(r"\belement\s*2\b", t_low):
        return "element_2"
    if re.search(r"\belement\s*3\b", t_low):
        return "element_3"
    if re.search(r"\belement\s*6\b", t_low):
        return "element_6"

    if "types and amount of scientific data" in t_low:
        return "element_1a"
    if "scientific data that will be preserved and shared" in t_low:
        return "element_1b"
    if "metadata, other relevant data, and associated documentation" in t_low:
        return "element_1c"

    if "repository where scientific data and metadata will be archived" in t_low:
        return "element_4a"
    if "how scientific data will be findable and identifiable" in t_low:
        return "element_4b"
    if "when and how long the scientific data will be made available" in t_low:
        return "element_4c"

    if "factors affecting subsequent access, distribution, or reuse" in t_low:
        return "element_5a"
    if "whether access to scientific data will be controlled" in t_low:
        return "element_5b"
    if "protections for privacy, rights, and confidentiality" in t_low:
        return "element_5c"

    # Optional: catch "Element 1:" etc
    if re.search(r"\belement\s*1\b", t_low):
        # If your markdown doesn't split 1a/1b/1c, return element_1 and handle separately
        return "element_1"

    return None

def get_element_columns(df: pd.DataFrame) -> list:
    cols = []
    for c in df.columns:
        cc = str(c).strip().lower()
        if re.match(r"^element_\d+[a-z]?$", cc):
            cols.append(cc)
    if not cols:
        raise ValueError("No element_* columns found in gold Excel.")
    return sorted(cols, key=sort_element_key)

In [20]:
# STEP 5 — Extract NIH Template Sections From Markdown (per title)
human_df = pd.read_excel(HUMAN_XLSX)
title_col = choose_title_column(human_df)
titles = human_df[title_col].dropna().astype(str).tolist()

def process_model_folder(model_dir: Path, content_col: str, out_csv: Path) -> pd.DataFrame:
    records = []

    for title in titles:
        md_path = find_md_by_excel_title_v3(model_dir, title)

        if md_path is None:
            records.append(
                {"dmp_title": title, "md_path": None, "Element title": None, content_col: None, "status": "missing_md"}
            )
            continue

        md_text = md_path.read_text(encoding="utf-8", errors="ignore")
        df_sec = extract_titles_and_text(md_text, content_col=content_col)

        if df_sec.empty:
            records.append(
                {"dmp_title": title, "md_path": str(md_path), "Element title": None, content_col: None, "status": "no_sections_found"}
            )
            continue

        for _, r in df_sec.iterrows():
            records.append(
                {"dmp_title": title, "md_path": str(md_path), "Element title": r["Element title"], content_col: r[content_col], "status": "ok"}
            )

    out_df = pd.DataFrame(records)
    out_df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"Saved: {out_csv} | rows={len(out_df)}")
    print(out_df["status"].value_counts(dropna=False))
    return out_df

process_model_folder(MODEL_DIR, "Generated_RAG_content", RAG_OUT_CSV)
print("Done: Step 5 (RAG)")

Saved: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results\filtered_RAG.csv | rows=312
status
ok    312
Name: count, dtype: int64
Done: Step 5 (RAG)


In [21]:
# STEP 6 — Merge NIH Reference With Model Output (sub-elements preserved)
ref = pd.read_excel(HUMAN_XLSX)
title_col = choose_title_column(ref)
ref = ref.rename(columns={title_col: "title"})
ref.columns = [str(c).strip().lower() for c in ref.columns]

ref["title_norm"] = ref["title"].apply(norm_text)
element_cols = get_element_columns(ref)

ref_long = ref[["title", "title_norm"] + element_cols].fillna("").melt(
    id_vars=["title", "title_norm"],
    value_vars=element_cols,
    var_name="Element number",
    value_name="NIH Value",
)
ref_long["Element number"] = ref_long["Element number"].str.lower().str.strip()

def load_model_filtered(path: Path, content_col: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["title_norm"] = df["dmp_title"].apply(norm_text)
    df["Element title"] = df["Element title"].fillna("").astype(str)

    df["Element number"] = df["Element title"].apply(map_section_title_to_element)
    df = df[df["Element number"].notna()].copy()
    df["Element number"] = df["Element number"].str.lower().str.strip()

    df = df.rename(columns={content_col: "Generated Content"})
    return df[["title_norm", "Element number", "Element title", "Generated Content"]]

def merge_and_save(sec_df: pd.DataFrame, out_path: Path):
    merged = ref_long.merge(sec_df, on=["title_norm", "Element number"], how="left")
    merged = merged[["title", "Element number", "NIH Value", "Element title", "Generated Content"]]
    merged.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved: {out_path} | rows={len(merged)}")

rag_sec = load_model_filtered(RAG_OUT_CSV, "Generated_RAG_content")
merge_and_save(rag_sec, RAG_MERGED)
print("Done: Step 6 (RAG)")

Saved: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results\merged_output_RAG.csv | rows=312
Done: Step 6 (RAG)


In [22]:
# STEP 7 — Collapse Sub-elements to Core Elements (1, 4, 5)
group_map = {
    "element_1": ["element_1a", "element_1b", "element_1c"],
    "element_4": ["element_4a", "element_4b", "element_4c"],
    "element_5": ["element_5a", "element_5b", "element_5c"],
}

def clean_text_block(x) -> str:
    s = safe_text(x).strip()
    s = re.sub(r"\n\s*\n+", "\n\n", s)
    return s

def combine_group_for_title(df_title: pd.DataFrame, new_element: str, group: list[str]) -> dict:
    g = df_title[df_title["Element number"].isin(group)].copy()
    g["Element number"] = pd.Categorical(g["Element number"], categories=group, ordered=True)
    g = g.sort_values("Element number")

    nih = "\n\n".join([clean_text_block(v) for v in g["NIH Value"].tolist() if clean_text_block(v)])
    gen = "\n\n".join([clean_text_block(v) for v in g["Generated Content"].tolist() if clean_text_block(v)])
    return {"Element number": new_element, "NIH Value": nih, "Generated Content": gen}

df = pd.read_csv(RAG_MERGED)

required = {"title", "Element number", "NIH Value", "Generated Content"}
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"{RAG_MERGED} missing columns: {missing}. Found: {list(df.columns)}")

df["Element number"] = df["Element number"].astype(str).str.strip().str.lower()

group_flat = [e for grp in group_map.values() for e in grp]
out_blocks = []

for title, df_title in df.groupby("title", dropna=False):
    df_title = df_title.copy()

    keep_df = df_title[~df_title["Element number"].isin(group_flat)].copy()
    keep_df = keep_df[["Element number", "NIH Value", "Generated Content"]]

    merged_rows = []
    for new_element, grp in group_map.items():
        merged_rows.append(combine_group_for_title(df_title, new_element, grp))

    final_title_df = pd.concat([keep_df, pd.DataFrame(merged_rows)], ignore_index=True)

    for col in ["NIH Value", "Generated Content"]:
        final_title_df[col] = final_title_df[col].apply(clean_text_block)

    final_title_df = final_title_df.sort_values(
        by="Element number", key=lambda s: s.map(sort_element_key)
    ).reset_index(drop=True)

    final_title_df.insert(0, "title", title)
    out_blocks.append(final_title_df)

final_df = pd.concat(out_blocks, ignore_index=True)
final_df.to_csv(RAG_CLEANED, index=False, encoding="utf-8")

print(f"Saved: {RAG_CLEANED} | rows={len(final_df)}")
print("Element counts:", final_df["Element number"].value_counts().to_dict())
print("Done: Step 7 (RAG)")

Saved: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results\merged_output_RAG_cleaned.csv | rows=156
Element counts: {'element_1': 26, 'element_2': 26, 'element_3': 26, 'element_4': 26, 'element_5': 26, 'element_6': 26}
Done: Step 7 (RAG)


In [23]:
# STEP 8 — Similarity Scoring (collapsed elements) + Sub-element Summary
sbert = SentenceTransformer("all-MiniLM-L6-v2")
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_similarity_tables(
    in_path: Path,
    model_name: str,
    write_folder_summary: bool,
    folder_summary_path: Path | None,
    raw_path: Path | None,
    element_summary_path: Path | None,
) -> pd.DataFrame:
    df = pd.read_csv(in_path)

    required = {"title", "Element number", "NIH Value", "Generated Content"}
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{in_path} missing columns: {missing}. Found: {list(df.columns)}")

    df["title"] = df["title"].apply(safe_text)
    df["Element number"] = df["Element number"].apply(safe_text).str.strip().str.lower()
    df["NIH Value"] = df["NIH Value"].apply(safe_text)
    df["Generated Content"] = df["Generated Content"].apply(safe_text)

    nih_texts = df["NIH Value"].tolist()
    gen_texts = df["Generated Content"].tolist()

    emb_nih = sbert.encode(nih_texts, convert_to_tensor=True, normalize_embeddings=True)
    emb_gen = sbert.encode(gen_texts, convert_to_tensor=True, normalize_embeddings=True)
    sbert_sims = (emb_nih * emb_gen).sum(dim=1).cpu().numpy().astype(float)

    rouge_recalls = [rougeL_recall(r, g, rouge) for r, g in zip(nih_texts, gen_texts)]

    raw = pd.DataFrame(
        {
            "Model": model_name,
            "DMP Title": df["title"],
            "Element number": df["Element number"],
            "SBERT_Similarity": sbert_sims,
            "ROUGE_L_Recall": rouge_recalls,
        }
    )

    if raw_path is not None:
        raw.to_csv(raw_path, index=False, encoding="utf-8")

    if write_folder_summary and folder_summary_path is not None:
        folder_summary = (
            raw.groupby(["Model", "DMP Title"], as_index=False)[["SBERT_Similarity", "ROUGE_L_Recall"]]
            .mean()
            .rename(columns={"DMP Title": "Folder"})
        )
        folder_summary.to_csv(folder_summary_path, index=False, encoding="utf-8")

    if element_summary_path is not None:
        element_summary = (
            raw.groupby(["Model", "Element number"], as_index=False)[["SBERT_Similarity", "ROUGE_L_Recall"]]
            .mean()
        )
        element_summary.to_csv(element_summary_path, index=False, encoding="utf-8")

    return raw


# Step 8a: collapsed elements (from cleaned file)
compute_similarity_tables(
    in_path=RAG_CLEANED,
    model_name="RAG",
    write_folder_summary=True,
    folder_summary_path=OUT_FOLDER,
    raw_path=OUT_RAW,
    element_summary_path=OUT_ELEMENT,
)

print("Saved Step 8a outputs:")
print(" ", OUT_FOLDER)
print(" ", OUT_RAW)
print(" ", OUT_ELEMENT)
print("Done: Step 8a (RAG)")


# Step 8b: sub-element summary (from non-cleaned merged file)
raw_sub = compute_similarity_tables(
    in_path=RAG_MERGED,
    model_name="RAG",
    write_folder_summary=False,
    folder_summary_path=None,
    raw_path=None,
    element_summary_path=None,
)

sub_element_summary = (
    raw_sub.groupby(["Model", "Element number"], as_index=False)[["SBERT_Similarity", "ROUGE_L_Recall"]]
    .mean()
    .sort_values(by="Element number", key=lambda s: s.map(sort_element_key))
    .reset_index(drop=True)
)

sub_element_summary.to_csv(OUT_SUB_ELEMENT, index=False, encoding="utf-8")
print("Saved:", OUT_SUB_ELEMENT)
print("Done: Step 8b (RAG)")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1674.55it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved Step 8a outputs:
  c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results\dmp_similarity_summary_RAG.csv
  c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results\element_similarity_raw_RAG.csv
  c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results\element_similarity_summary_RAG.csv
Done: Step 8a (RAG)
Saved: c:\Users\Nahid\dmpchef\notebook_DMP_RAG\3-Output_experiemnt_RAG_v3_NIH_all\run_20260304_145748\evaluation_results\sub_element_similarity_summary_RAG.csv
Done: Step 8b (RAG)
